# 实现端到端的模型方案


In [ ]:
for _, series_uid in series_iter:
    ct = getCt(series_uid)

    mask_a = self.segmentCt(ct, series_uid)
    candadateInfo_list = self.groupSegmentationOutput(series_uid, ct, mask_a)

    calsssifications_list = self.classsifyCandidates(ct, candidateInfo_list)
    

In [ ]:
def segmentCt(self, ct, series_uid):
    with torch.no_grad():
        output_a = np.zeros_likes(ct.hus_a, dtype=np.float32)

        seg_dl = self.initSegmentationsD1(series_uid)

        for input_t, _, _, slice_ndx_list in seg_dl:
            input_g = input_t.to(self.device)
            prediction_g = self.seg_model(input_g)

            for i, slice_ndx in enumerate(slice_ndx_list):
                output_a[slice_ndx] = prediction_g[i].cpu().numpy()
            
            mask_a = output_a > 0.5
            mask_a = morphology.binary_erossion(mask_a, iterations = 1)

    return mask_a

In [ ]:
#scipy.ndimage.measurements
# measurements.label  联通区域 上下左右 联通的 会按顺序标出
 
#001100001100
#000100001100
#000000100000

#001100002200
#000100002200
#000000300000

#measurements.center_of_mass

In [ ]:
def groupSegmentationOutput(self, series_uid, ct, clean_a):
    
    candidateLabel_a, candidate_count = measurements.label(clean_a)
    centerIrc_list = measurements.center_of_mass(
        ct.hu_a.clip(-1000, 1000) + 1001,
        labels = candidateLabel_a,
        index = np.arrange(1, candidate_count+1)
    )
    
    candidateInfo_list = []
    for i, center_irc in enumerate(centerIrc_list):
        center_xyz = irc2xyz(
            center_irc,
            ct.origin_xyz,
            ct.vxSize_xyz,
            ct.direction_a,
        )
        
        candidateInfo_tup = CandidateInfoTuple(False, False, False, 0.0, series_uid, center_xyz)
        candidateInfo_list.append(candidateInfo_tup)
        
    return candidateInfo_list

In [ ]:
def classifyCandidates(self, ct, candidateInfo_list):
    
    cls_dl = self.initClassificationDl(candidateInfo_list)
    classification_list = []
    
    for batch_ndx, batch_tup in enumerate(cls_dl):
        input_t, _, _, series_list, center_list = batch_tup
        
        input_g = input_t.to(self.device)
        
        with torch.no_grad():
            _, probability_nodule_g = self.cls_model(input_g)
            
            if self.malignancy_model is not None:
                _, probability_mal_g = self.malignancy_model(input_g)
            else:
                probability_mal_g = torch.zeros_like(probability_nodule_g)
                
        zip_iter = zip(center_list, probability_nodule_g[:,1].tolist(), probability_mal_g[:,1].tolist())
        
        for center_irc, prob_nodule, prob_mal in zip_iter:
            center_xyz = irc2xyz(
            center_irc,
            direction_a=ct.direction_a,
            origin_xyz=ct.origin_xyz,
            vxSize_xyz=ct.vxSize_xyz
        )
            cls_tup = (prob_nodule, prob_mal, center_xyz, center_irc)
            classification_list.append(cls_tup)
        return classification_list